In [50]:
pip install transformers==4.57.6 transformer_lens==2.17.0 huggingface_hub==0.36.1 accelerate==1.12.0 tqdm


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [51]:
import huggingface_hub
assert huggingface_hub.__version__ == "0.36.1", \
    f"RESTART KERNEL! Got {huggingface_hub.__version__}, need 0.36.1"

In [52]:
import transformers
import transformer_lens
import torch

print(f'torch={torch.__version__}, transformers={transformers.__version__}, CUDA={torch.cuda.is_available()}')

torch=2.8.0+cu128, transformers=4.57.6, CUDA=True


## Quick Sanity Check

Before running the full pipeline, test one model load:

In [53]:
from transformers import AutoModelForCausalLM
import torch
m = AutoModelForCausalLM.from_pretrained('EleutherAI/pythia-160m-deduped', revision='step0', use_safetensors=False)
print(f'✅ Model loaded, params: {sum(p.numel() for p in m.parameters())/1e6:.0f}M')
print(f'GPU: {torch.cuda.get_device_name(0)}')

✅ Model loaded, params: 162M
GPU: Tesla T4


Run Scoring Tests

In [54]:
%cd attention-binding-a11y
!python tests/test_behavioral.py

[Errno 2] No such file or directory: 'attention-binding-a11y'
/teamspace/studios/this_studio/attention-binding-a11y


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/IPython/core/magics/osm.py:393: UserWarning: using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})


✅ Recognition scoring tests passed
✅ Generation scoring tests passed
✅ Contradiction penalty tests passed
✅ evaluate_output recognition tests passed
✅ evaluate_output generation tests passed
✅ Invalid task error tests passed

✅ All behavioral tests passed!


## Run the Pipeline — Single Checkpoint First

This will:
* Download pythia-160m-deduped at step0
* Run all 12 prompts
* Save results to `data/results/behavioral/160m_step0_behavioral.jsonl`

In [55]:
!python3 src/eval_behavior.py 160m step0

Loading 160m step0...
Loaded pretrained model EleutherAI/pythia-160m-deduped into HookedTransformer
Evaluating: 100%|███████████████████████████████| 12/12 [00:03<00:00,  3.54it/s]
Saved 12 results to data/results/behavioral/160m_step0_behavioral.jsonl
Complete: data/results/behavioral/160m_step0_behavioral.jsonl


Check the output. 
Look for: `text_out` values that are coherent, `score`values that vary.

In [56]:
%cd attention-binding-a11y
!head data/results/behavioral/160m_step0_behavioral.jsonl

[Errno 2] No such file or directory: 'attention-binding-a11y'
/teamspace/studios/this_studio/attention-binding-a11y


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/IPython/core/magics/osm.py:393: UserWarning: using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})


{"model": "pythia-160m-deduped", "checkpoint": "step0", "term": "screen reader", "task": "recognition", "prompt_id": "rec_001", "prompt_template": "A screen reader is primarily used by: A) Blind users B) Colorblind users C) Deaf users D) Mobility impaired users", "text_out": "predicted=B", "full_output": "A screen reader is primarily used by: A) Blind users B) Colorblind users C) Deaf users D) Mobility impaired users", "is_correct": false, "score": 0.0, "eval_method": "logprob_rank", "predicted_idx": 1, "log_probs": [-11.058307647705078, -10.756403923034668, -10.893818855285645, -10.911664247512817]}
{"model": "pythia-160m-deduped", "checkpoint": "step0", "term": "screen reader", "task": "recognition", "prompt_id": "rec_002", "prompt_template": "Which group benefits most from screen readers? A) People with visual impairments B) People with hearing loss C) People with motor disabilities D) People with cognitive disabilities", "text_out": "predicted=C", "full_output": "Which group benefi

In [57]:
%cd attention-binding-a11y
!head data/results/behavioral/160m_step143000_behavioral.jsonl

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/IPython/core/magics/osm.py:393: UserWarning: using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})


[Errno 2] No such file or directory: 'attention-binding-a11y'
/teamspace/studios/this_studio/attention-binding-a11y


{"model": "pythia-160m-deduped", "checkpoint": "step143000", "term": "screen reader", "task": "recognition", "prompt_id": "rec_001", "prompt_template": "A screen reader is primarily used by: A) Blind users B) Colorblind users C) Deaf users D) Mobility impaired users", "text_out": "predicted=D", "full_output": "A screen reader is primarily used by: A) Blind users B) Colorblind users C) Deaf users D) Mobility impaired users", "is_correct": false, "score": 0.0, "eval_method": "logprob_rank", "predicted_idx": 3, "log_probs": [-6.54019064642489, -4.012171741575003, -5.882156778126955, -3.2200761092826724]}
{"model": "pythia-160m-deduped", "checkpoint": "step143000", "term": "screen reader", "task": "recognition", "prompt_id": "rec_002", "prompt_template": "Which group benefits most from screen readers? A) People with visual impairments B) People with hearing loss C) People with motor disabilities D) People with cognitive disabilities", "text_out": "predicted=A", "full_output": "Which group 

## Run Full 160m Sweep (8 checkpoints)

**Estimated time:** ~2–5 min per checkpoint for 160m (mostly download on first run).

In [58]:
%%bash
for step in step0 step15000 step30000 step60000 step90000 step120000 step140000 step143000; do
    echo "=== Running 160m $step ==="
    python3 src/eval_behavior.py "160m" "$step"
done

=== Running 160m step0 ===
Loading 160m step0...
Loaded pretrained model EleutherAI/pythia-160m-deduped into HookedTransformer


Evaluating: 100%|██████████| 12/12 [00:03<00:00,  3.63it/s]


Saved 12 results to data/results/behavioral/160m_step0_behavioral.jsonl
Complete: data/results/behavioral/160m_step0_behavioral.jsonl
=== Running 160m step15000 ===
Loading 160m step15000...
Loaded pretrained model EleutherAI/pythia-160m-deduped into HookedTransformer


Evaluating: 100%|██████████| 12/12 [00:03<00:00,  3.47it/s]


Saved 12 results to data/results/behavioral/160m_step15000_behavioral.jsonl
Complete: data/results/behavioral/160m_step15000_behavioral.jsonl
=== Running 160m step30000 ===
Loading 160m step30000...
Loaded pretrained model EleutherAI/pythia-160m-deduped into HookedTransformer


...

Saved 12 results to data/results/behavioral/160m_step120000_behavioral.jsonl
Complete: data/results/behavioral/160m_step120000_behavioral.jsonl
=== Running 160m step140000 ===
Loading 160m step140000...
Loaded pretrained model EleutherAI/pythia-160m-deduped into HookedTransformer


Evaluating: 100%|██████████| 12/12 [00:03<00:00,  3.42it/s]


Saved 12 results to data/results/behavioral/160m_step140000_behavioral.jsonl
Complete: data/results/behavioral/160m_step140000_behavioral.jsonl
=== Running 160m step143000 ===
Loading 160m step143000...
Loaded pretrained model EleutherAI/pythia-160m-deduped into HookedTransformer


Evaluating: 100%|██████████| 12/12 [00:03<00:00,  3.68it/s]


Saved 12 results to data/results/behavioral/160m_step143000_behavioral.jsonl
Complete: data/results/behavioral/160m_step143000_behavioral.jsonl


## Scale to 1b and 2.8b

**Estimated time:**

* 1b: ~5–10 min per checkpoint
* 2.8b: ~10–15 min per checkpoint (37s load + generation time)

In [59]:
%%bash
for size in 1b 2.8b; do
    for step in step0 step15000 step30000 step60000 step90000 step120000 step140000 step143000; do
        echo "=== Running $size $step ==="
        python3 src/eval_behavior.py $size $step
    done
done

=== Running 1b step0 ===
Loading 1b step0...
Loaded pretrained model EleutherAI/pythia-1b-deduped into HookedTransformer


Evaluating: 100%|██████████| 12/12 [00:04<00:00,  2.57it/s]


Saved 12 results to data/results/behavioral/1b_step0_behavioral.jsonl
Complete: data/results/behavioral/1b_step0_behavioral.jsonl
=== Running 1b step15000 ===
Loading 1b step15000...
Loaded pretrained model EleutherAI/pythia-1b-deduped into HookedTransformer


Evaluating: 100%|██████████| 12/12 [00:04<00:00,  2.49it/s]


Saved 12 results to data/results/behavioral/1b_step15000_behavioral.jsonl
Complete: data/results/behavioral/1b_step15000_behavioral.jsonl
=== Running 1b step30000 ===
Loading 1b step30000...
Loaded pretrained model EleutherAI/pythia-1b-deduped into HookedTransformer


...

Saved 12 results to data/results/behavioral/2.8b_step120000_behavioral.jsonl
Complete: data/results/behavioral/2.8b_step120000_behavioral.jsonl
=== Running 2.8b step140000 ===
Loading 2.8b step140000...
Loaded pretrained model EleutherAI/pythia-2.8b-deduped into HookedTransformer


Evaluating: 100%|██████████| 12/12 [00:12<00:00,  1.04s/it]


Saved 12 results to data/results/behavioral/2.8b_step140000_behavioral.jsonl
Complete: data/results/behavioral/2.8b_step140000_behavioral.jsonl
=== Running 2.8b step143000 ===
Loading 2.8b step143000...
Loaded pretrained model EleutherAI/pythia-2.8b-deduped into HookedTransformer


Evaluating: 100%|██████████| 12/12 [00:12<00:00,  1.00s/it]


Saved 12 results to data/results/behavioral/2.8b_step143000_behavioral.jsonl
Complete: data/results/behavioral/2.8b_step143000_behavioral.jsonl


## Verify Results

In [60]:
%%bash
# Count result files (should be 24: 3 sizes × 8 checkpoints)
ls -la data/results/behavioral/ | wc -l

# Quick accuracy check
python3 -c "
import sys
sys.path.insert(0, 'src')
from eval_behavior import aggregate_behavioral_results
results = aggregate_behavioral_results()
for key, val in sorted(results.items()):
    model, step, term, task = key
    print(f'{model:30s} {step:12s} {term:15s} {task:12s} acc={val[\"accuracy\"]:.2f} score={val[\"mean_score\"]:.2f} n={val[\"n\"]}')
"

27


pythia-160m-deduped            step0        alt text        generation   acc=0.00 score=0.00 n=2
pythia-160m-deduped            step0        alt text        recognition  acc=0.00 score=0.00 n=2
pythia-160m-deduped            step0        screen reader   generation   acc=0.00 score=0.00 n=2
pythia-160m-deduped            step0        screen reader   recognition  acc=0.00 score=0.00 n=2
pythia-160m-deduped            step0        skip link       generation   acc=0.00 score=0.00 n=2
pythia-160m-deduped            step0        skip link       recognition  acc=0.50 score=0.50 n=2
pythia-160m-deduped            step120000   alt text        generation   acc=0.00 score=0.33 n=2
pythia-160m-deduped            step120000   alt text        recognition  acc=1.00 score=1.00 n=2
pythia-160m-deduped            step120000   screen reader   generation   acc=1.00 score=1.00 n=2
pythia-160m-deduped            step120000   screen reader   recognition  acc=0.00 score=0.00 n=2
pythia-160m-deduped           

## Check learning curve across 8 checkpoints

In [61]:
%%bash
python3 -c "
import json, os

steps = ['step0', 'step15000', 'step30000', 'step60000', 'step90000', 'step120000', 'step140000', 'step143000']
print(f'{'Step':>12s}  {'Rec Acc':>8s}  {'Gen Mean':>8s}  {'Overall':>8s}')
print('-' * 45)

for step in steps:
    path = f'data/results/behavioral/160m_{step}_behavioral.jsonl'
    if not os.path.exists(path):
        print(f'{step:>12s}  MISSING')
        continue
    rec_correct = 0; rec_total = 0; gen_scores = []
    with open(path) as f:
        for line in f:
            r = json.loads(line)
            if r['task'] == 'recognition':
                rec_total += 1
                if r['is_correct']: rec_correct += 1
            else:
                gen_scores.append(r['score'])
    rec_acc = rec_correct / rec_total if rec_total else 0
    gen_mean = sum(gen_scores) / len(gen_scores) if gen_scores else 0
    overall = (rec_acc + gen_mean) / 2
    print(f'{step:>12s}  {rec_acc:>8.1%}  {gen_mean:>8.4f}  {overall:>8.4f}')
"

        Step   Rec Acc  Gen Mean   Overall
---------------------------------------------
       step0     16.7%    0.0000    0.0833
   step15000      0.0%    0.3333    0.1667
   step30000     16.7%    0.6667    0.4167
   step60000     16.7%    0.5556    0.3611
   step90000     50.0%    0.5555    0.5278
  step120000     66.7%    0.5555    0.6111
  step140000     66.7%    0.5555    0.6111
  step143000     50.0%    0.5000    0.5000
